# House Price Prediction with Linear Regression

**Objective:** Build and evaluate a linear regression model that predicts house prices
based on features such as area, location, number of rooms, and age.

**Dataset:** Download the **"House Prices - Advanced Regression Techniques"** dataset
from Kaggle: https://www.kaggle.com/c/house-prices-advanced-regression-techniques/data
(or search "house price prediction dataset" on Kaggle for the Ames Housing Dataset).

Save the training file as `train.csv` in the same folder as this notebook before running.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

sns.set_style('whitegrid')
%matplotlib inline

## 2. Load Data & Initial EDA

In [ ]:
df = pd.read_csv('train.csv')
print(df.shape)
df.head()

In [ ]:
# Null check
nulls = df.isnull().sum()
nulls = nulls[nulls > 0].sort_values(ascending=False)
print(nulls)

In [ ]:
# Descriptive stats and target distribution
df['SalePrice'].describe()

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df['SalePrice'], kde=True)
plt.title('Distribution of SalePrice')
plt.show()

**Observation:** Note here whether SalePrice is right-skewed. If so, a log
transform of the target often improves linear regression performance — try
`np.log1p(df['SalePrice'])` and compare results later.

## 3. Feature Selection

**Reasoning:** We start with a compact set of features that are commonly strong
predictors of house price: living area, overall quality, number of rooms/bathrooms,
garage capacity, and age of the house. This keeps the model interpretable while
still capturing the main price drivers.

In [ ]:
features = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF',
            'FullBath', 'YearBuilt', 'YearRemodAdd', 'LotArea', 'BedroomAbvGr']
target = 'SalePrice'

data = df[features + [target]].copy()
data.isnull().sum()

## 4. Handle Missing Values & Encode Categoricals

In [ ]:
# Impute any remaining numeric nulls with median
for col in features:
    if data[col].isnull().any():
        data[col] = data[col].fillna(data[col].median())

# Add house age as an engineered feature
data['HouseAge'] = df['YrSold'] - data['YearBuilt'] if 'YrSold' in df.columns else 2024 - data['YearBuilt']

data.isnull().sum().sum()  # should be 0

## 5. Correlation Heatmap

In [ ]:
plt.figure(figsize=(10,8))
corr = data.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

## 6. Train/Test Split

In [ ]:
X = data.drop(columns=[target])
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

## 7. Train Linear Regression Model

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

## 8. Evaluation

In [ ]:
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MSE:  {mse:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R2:   {r2:.4f}")

## 9. Actual vs Predicted

In [ ]:
plt.figure(figsize=(7,7))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Price')
plt.ylabel('Predicted Price')
plt.title('Actual vs Predicted House Prices')
plt.show()

## 10. Residual Plot

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8,5))
sns.scatterplot(x=y_pred, y=residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted Price')
plt.ylabel('Residual')
plt.title('Residual Plot')
plt.show()

## 11. Coefficient Analysis

In [ ]:
coefs = pd.Series(model.coef_, index=X.columns).sort_values(key=abs, ascending=False)
print(coefs)

plt.figure(figsize=(8,5))
coefs.plot(kind='barh')
plt.title('Feature Coefficients (impact on price)')
plt.gca().invert_yaxis()
plt.show()

## 12. Bonus: Ridge & Lasso Comparison

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ridge = Ridge(alpha=1.0).fit(X_train_scaled, y_train)
lasso = Lasso(alpha=1.0).fit(X_train_scaled, y_train)

for name, m in [('Linear', model), ('Ridge', ridge), ('Lasso', lasso)]:
    if name == 'Linear':
        pred = model.predict(X_test)
    else:
        pred = m.predict(X_test_scaled)
    print(f"{name:8s} R2: {r2_score(y_test, pred):.4f}  RMSE: {np.sqrt(mean_squared_error(y_test, pred)):,.2f}")

## Conclusion

Write 2-3 sentences here summarizing:
- Which features had the strongest positive/negative impact on price
- How well the model performed (R² / RMSE)
- Whether Ridge/Lasso regularization improved generalization over plain Linear Regression